# 📈 Gold Price Prediction using Random Forest
**Project by Yuwa** — Predicts gold futures closing prices using historical data and machine learning.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

DATA_FILE = 'gold_price_data.csv'

def load_or_download():
    """Download gold data if not present, or load from disk.
    Handles the newer yfinance MultiIndex column format.
    """
    if os.path.exists(DATA_FILE):
        # Try reading the existing CSV; if Date is not valid, re-download
        try:
            df = pd.read_csv(DATA_FILE, index_col='Date', parse_dates=True)
            # Sanity check: must have Close column
            if 'Close' not in df.columns:
                raise ValueError('Bad CSV format, re-downloading...')
            print(f'{DATA_FILE} loaded from disk. Shape: {df.shape}')
            return df
        except (ValueError, KeyError) as e:
            print(f'Existing file is corrupt or in old format ({e}). Re-downloading...')
            os.remove(DATA_FILE)

    print('Downloading historical gold prices from Yahoo Finance...')
    df = yf.download('GC=F', start='2015-01-01', end='2023-12-31')

    if df.empty:
        raise RuntimeError('Download failed. Check your internet connection.')

    # Newer yfinance returns MultiIndex columns (Price, Ticker) — flatten them
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel('Ticker')

    df.to_csv(DATA_FILE)
    print(f'Data saved to {DATA_FILE}. Shape: {df.shape}')
    return df

gold_data = load_or_download()
gold_data.head()

## 🔍 Exploratory Data Analysis

In [ ]:
print('Dataset Info:')
print(gold_data.info())
print('\nBasic Statistics:')
print(gold_data.describe())

# Plot closing price over time
plt.figure(figsize=(14, 5))
plt.plot(gold_data.index, gold_data['Close'], color='gold', linewidth=1.5)
plt.title('Gold Futures Closing Price (2015-2023)', fontsize=14)
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## ⚙️ Feature Engineering & Model Training

In [ ]:
# Feature Engineering
df = gold_data.copy().dropna()
df['Prev_Close'] = df['Close'].shift(1)
df['MA_5']  = df['Close'].rolling(window=5).mean().shift(1)
df['MA_10'] = df['Close'].rolling(window=10).mean().shift(1)
df = df.dropna()

features = ['Open', 'High', 'Low', 'Volume', 'Prev_Close', 'MA_5', 'MA_10']
X = df[features]
y = df['Close']

# Chronological split (no shuffling for time-series)
split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f'Training samples: {len(X_train)} | Test samples: {len(X_test)}')

# Train Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print('Model trained!')

## 📊 Evaluation & Results

In [ ]:
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae  = mean_absolute_error(y_test, predictions)

print(f'✅ RMSE : {rmse:.2f}')
print(f'✅ MAE  : {mae:.2f}')

# Plot Actual vs Predicted
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(y_test.values, label='Actual',    color='gold',  linewidth=1.5)
axes[0].plot(predictions,   label='Predicted', color='steelblue', linewidth=1.5, alpha=0.8)
axes[0].set_title('Actual vs Predicted Gold Price')
axes[0].set_xlabel('Test Samples')
axes[0].set_ylabel('Price (USD)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Feature Importances
importances = model.feature_importances_
indices = np.argsort(importances)
axes[1].barh(range(len(indices)), importances[indices], color='steelblue', align='center')
axes[1].set_yticks(range(len(indices)))
axes[1].set_yticklabels([X.columns[i] for i in indices])
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Relative Importance')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results.png', dpi=150)
plt.show()
print('Plot saved to results.png')